# Dynamic Partition Pruning
- Optimization technique
- Runtime filter  
- Data should be partitioned

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.master("local[*]")\
        .appName("dynamicPartitionPruningApplicationTest")\
        .getOrCreate()

spark

Reading the retails dataset and saving into parqet file

In [2]:
# # Reading the retails dataset and saving into parqet file
# retailDf = spark.read.format("csv")\
#     .option("header", "true")\
#     .option("inferSchema", "true")\
#     .load("data/online-retail-dataset.csv")

# retailDf=retailDf.withColumn("date", to_date(retailDf["InvoiceDate"],  "M/d/yyyy H:mm"))

# retailDf.write.format("parquet")\
#     .mode("overwrite")\
#     .partitionBy("date")\
#     .save("D:/BIG_DATA/sparkDatabase/retails")

### Read the parquet file

In [3]:
retailsDf = spark.read.format("parquet").load("D:/BIG_DATA/sparkDatabase/retails")

### creating the date dimension table

In [4]:
date_dim_table = retailsDf.select("date").distinct()
date_dim_table = date_dim_table.withColumn("year", year(date_dim_table["date"]))

date_dim_table.select("year").distinct().show()

+----+
|year|
+----+
|2010|
|2011|
+----+



### Testing the Dynamic Partition Pruning

In [5]:
retailsDf.join(date_dim_table, retailsDf["date"] == date_dim_table["date"], "inner")\
    .filter(date_dim_table["year"] == 2010).show()

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+----------+----------+----+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|      date|      date|year|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+----------+----------+----+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|12/6/2010 8:34|     2.95|     15987|United Kingdom|2010-12-06|2010-12-06|2010|
|   537226|    21713|CITRONELLA CANDLE...|       8|12/6/2010 8:34|      2.1|     15987|United Kingdom|2010-12-06|2010-12-06|2010|
|   537226|    22927|GREEN GIANT GARDE...|       2|12/6/2010 8:34|     5.95|     15987|United Kingdom|2010-12-06|2010-12-06|2010|
|   537226|    20802|SMALL GLASS SUNDA...|       6|12/6/2010 8:34|     1.65|     15987|United Kingdom|2010-12-06|2010-12-06|2010|
|   537226|    22052|VINTAGE CARAVAN G...|      25|12/6/2010 8:34|     0.42|     15987|Uni

In above code snipet partition pruning applied, we can go into spark UI and check under the spark sql detail section

```python
Location: InMemoryFileIndex [file:/D:/BIG_DATA/sparkDatabase/retails]
PartitionFilters: [(year(date#8) = 2010), isnotnull(date#8), dynamicpruningexpression(date#8 IN dynamicpruning#94)]
ReadSchema: struct<InvoiceNo:string,StockCode:string,Description:string,Quantity:int,InvoiceDate:string,UnitPrice:double,CustomerID:int,Country:string>
```